# Diagnóstico Bronze — steam_games

Tabla analizada: `steam_game_intelligence.bronze.steam_games`. Revisa nulos, duplicados, rangos imposibles y snapshots diarios. El esquema esperado se basa en el archivo de definición que compartiste.

> Ejecuta las celdas en orden. Estas consultas son de diagnóstico: no modifican las tablas Bronze.

## 1. Volumen y ventana de ingesta

In [0]:
%sql
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT app_id) AS distinct_app_ids,
  MIN(_ingested_at) AS first_ingested_at,
  MAX(_ingested_at) AS last_ingested_at,
  COUNT_IF(app_id IS NULL) AS null_app_id_rows,
  COUNT_IF(name IS NULL OR TRIM(name) = '') AS missing_name_rows
FROM steam_game_intelligence.bronze.steam_games;

## 2. Nulos por columna
La salida muestra únicamente columnas con nulos. En campos de texto también conviene distinguir NULL de cadenas vacías.

In [0]:
%sql
SELECT column_name, null_count
FROM (
  SELECT
    COUNT_IF(app_id IS NULL) AS app_id,
    COUNT_IF(name IS NULL) AS name,
    COUNT_IF(release_date IS NULL) AS release_date,
    COUNT_IF(price IS NULL) AS price,
    COUNT_IF(price_status IS NULL) AS price_status,
    COUNT_IF(estimated_owners IS NULL) AS estimated_owners,
    COUNT_IF(developers IS NULL) AS developers,
    COUNT_IF(publishers IS NULL) AS publishers,
    COUNT_IF(genres IS NULL) AS genres,
    COUNT_IF(categories IS NULL) AS categories,
    COUNT_IF(tags IS NULL) AS tags,
    COUNT_IF(positive IS NULL) AS positive,
    COUNT_IF(negative IS NULL) AS negative,
    COUNT_IF(recommendations IS NULL) AS recommendations,
    COUNT_IF(peak_ccu IS NULL) AS peak_ccu,
    COUNT_IF(metacritic_score IS NULL) AS metacritic_score,
    COUNT_IF(user_score IS NULL) AS user_score,
    COUNT_IF(average_playtime_forever IS NULL) AS average_playtime_forever,
    COUNT_IF(median_playtime_forever IS NULL) AS median_playtime_forever,
    COUNT_IF(average_playtime_2weeks IS NULL) AS average_playtime_2weeks,
    COUNT_IF(median_playtime_2weeks IS NULL) AS median_playtime_2weeks,
    COUNT_IF(short_description IS NULL) AS short_description,
    COUNT_IF(about_the_game IS NULL) AS about_the_game,
    COUNT_IF(detailed_description IS NULL) AS detailed_description,
    COUNT_IF(notes IS NULL) AS notes,
    COUNT_IF(achievements IS NULL) AS achievements,
    COUNT_IF(dlc_count IS NULL) AS dlc_count,
    COUNT_IF(packages IS NULL) AS packages,
    COUNT_IF(supported_languages IS NULL) AS supported_languages,
    COUNT_IF(full_audio_languages IS NULL) AS full_audio_languages,
    COUNT_IF(windows IS NULL) AS windows,
    COUNT_IF(mac IS NULL) AS mac,
    COUNT_IF(linux IS NULL) AS linux,
    COUNT_IF(steam_store_available IS NULL) AS steam_store_available,
    COUNT_IF(steam_spy_available IS NULL) AS steam_spy_available,
    COUNT_IF(_ingested_at IS NULL) AS _ingested_at,
    COUNT_IF(_source IS NULL) AS _source
  FROM steam_game_intelligence.bronze.steam_games
) UNPIVOT (null_count FOR column_name IN (
  app_id, name, release_date, price, price_status, estimated_owners,
  developers, publishers, genres, categories, tags, positive, negative,
  recommendations, peak_ccu, metacritic_score, user_score,
  average_playtime_forever, median_playtime_forever, average_playtime_2weeks,
  median_playtime_2weeks, short_description, about_the_game, detailed_description,
  notes, achievements, dlc_count, packages, supported_languages,
  full_audio_languages, windows, mac, linux, steam_store_available,
  steam_spy_available, _ingested_at, _source
))
WHERE null_count > 0
ORDER BY null_count DESC;

## 3. Claves repetidas
Como la tabla se ingiere en append diario, que `app_id` aparezca varias veces puede ser un snapshot esperado, no necesariamente un duplicado incorrecto. La segunda consulta separa repetición global de repetición dentro de una misma carga.

In [0]:
%sql
SELECT app_id, COUNT(*) AS row_count,
       COUNT(DISTINCT CAST(_ingested_at AS DATE)) AS ingestion_days,
       MIN(_ingested_at) AS first_seen,
       MAX(_ingested_at) AS last_seen
FROM steam_game_intelligence.bronze.steam_games
WHERE app_id IS NOT NULL
GROUP BY app_id
HAVING COUNT(*) > 1
ORDER BY row_count DESC
LIMIT 100;

In [0]:
%sql
SELECT app_id, CAST(_ingested_at AS DATE) AS ingestion_date,
       COUNT(*) AS row_count
FROM steam_game_intelligence.bronze.steam_games
GROUP BY app_id, CAST(_ingested_at AS DATE)
HAVING COUNT(*) > 1
ORDER BY row_count DESC
LIMIT 100;

## 4. Valores fuera de rango o sospechosos
Los límites son reglas de diagnóstico, no afirmaciones de que todo valor fuera de ellos sea inválido.

In [0]:
%sql
SELECT
  COUNT_IF(app_id <= 0) AS invalid_app_id,
  COUNT_IF(price < 0) AS negative_price,
  COUNT_IF(positive < 0) AS negative_positive_reviews,
  COUNT_IF(negative < 0) AS negative_negative_reviews,
  COUNT_IF(recommendations < 0) AS negative_recommendations,
  COUNT_IF(peak_ccu < 0) AS negative_peak_ccu,
  COUNT_IF(metacritic_score < 0 OR metacritic_score > 100) AS metacritic_out_of_range,
  COUNT_IF(user_score < 0 OR user_score > 100) AS user_score_out_of_range,
  COUNT_IF(achievements < 0) AS negative_achievements,
  COUNT_IF(dlc_count < 0) AS negative_dlc_count,
  COUNT_IF(release_date > CURRENT_DATE()) AS future_release_date,
  COUNT_IF(_ingested_at IS NULL) AS missing_ingestion_timestamp
FROM steam_game_intelligence.bronze.steam_games;

In [0]:
%sql
SELECT app_id, name, release_date, price, positive, negative,
       recommendations, peak_ccu, metacritic_score, user_score,
       achievements, dlc_count, _ingested_at
FROM steam_game_intelligence.bronze.steam_games
WHERE app_id <= 0
   OR price < 0
   OR positive < 0 OR negative < 0
   OR recommendations < 0 OR peak_ccu < 0
   OR metacritic_score < 0 OR metacritic_score > 100
   OR user_score < 0 OR user_score > 100
   OR achievements < 0 OR dlc_count < 0
   OR release_date > CURRENT_DATE()
   OR name IS NULL OR TRIM(name) = ''
LIMIT 200;

## 5. Cadenas vacías y normalización básica

In [0]:
%sql
SELECT
  COUNT_IF(name IS NOT NULL AND TRIM(name) = '') AS blank_name,
  COUNT_IF(developers IS NOT NULL AND TRIM(developers) = '') AS blank_developers,
  COUNT_IF(publishers IS NOT NULL AND TRIM(publishers) = '') AS blank_publishers,
  COUNT_IF(genres IS NOT NULL AND TRIM(genres) = '') AS blank_genres,
  COUNT_IF(categories IS NOT NULL AND TRIM(categories) = '') AS blank_categories,
  COUNT_IF(_source IS NOT NULL AND TRIM(_source) = '') AS blank_source
FROM steam_game_intelligence.bronze.steam_games;

## 6. Variaciones entre snapshots
Detecta app_id cuyo nombre o precio cambió entre ingestas. Es información útil para decidir cómo construir Silver; no se debe borrar el historial sin definir primero el objetivo.

In [0]:
%sql
SELECT app_id,
       COUNT(DISTINCT name) AS distinct_names,
       COUNT(DISTINCT price) AS distinct_prices,
       COUNT(DISTINCT release_date) AS distinct_release_dates,
       COUNT(DISTINCT genres) AS distinct_genre_values,
       COUNT(*) AS snapshots
FROM steam_game_intelligence.bronze.steam_games
WHERE app_id IS NOT NULL
GROUP BY app_id
HAVING COUNT(DISTINCT name) > 1
    OR COUNT(DISTINCT price) > 1
    OR COUNT(DISTINCT release_date) > 1
    OR COUNT(DISTINCT genres) > 1
ORDER BY snapshots DESC
LIMIT 100;

In [0]:
%python
from datetime import datetime
import os

table_name = 'steam_game_intelligence.bronze.steam_games'
notebook_name = '01_diagnostico_steam_games'
queries = [('Consulta de celda SQL 1', "SELECT\n  COUNT(*) AS total_rows,\n  COUNT(DISTINCT app_id) AS distinct_app_ids,\n  MIN(_ingested_at) AS first_ingested_at,\n  MAX(_ingested_at) AS last_ingested_at,\n  COUNT_IF(app_id IS NULL) AS null_app_id_rows,\n  COUNT_IF(name IS NULL OR TRIM(name) = '') AS missing_name_rows\nFROM steam_game_intelligence.bronze.steam_games;"), ('Consulta de celda SQL 2', 'SELECT column_name, null_count\nFROM (\n  SELECT\n    COUNT_IF(app_id IS NULL) AS app_id,\n    COUNT_IF(name IS NULL) AS name,\n    COUNT_IF(release_date IS NULL) AS release_date,\n    COUNT_IF(price IS NULL) AS price,\n    COUNT_IF(price_status IS NULL) AS price_status,\n    COUNT_IF(estimated_owners IS NULL) AS estimated_owners,\n    COUNT_IF(developers IS NULL) AS developers,\n    COUNT_IF(publishers IS NULL) AS publishers,\n    COUNT_IF(genres IS NULL) AS genres,\n    COUNT_IF(categories IS NULL) AS categories,\n    COUNT_IF(tags IS NULL) AS tags,\n    COUNT_IF(positive IS NULL) AS positive,\n    COUNT_IF(negative IS NULL) AS negative,\n    COUNT_IF(recommendations IS NULL) AS recommendations,\n    COUNT_IF(peak_ccu IS NULL) AS peak_ccu,\n    COUNT_IF(metacritic_score IS NULL) AS metacritic_score,\n    COUNT_IF(user_score IS NULL) AS user_score,\n    COUNT_IF(average_playtime_forever IS NULL) AS average_playtime_forever,\n    COUNT_IF(median_playtime_forever IS NULL) AS median_playtime_forever,\n    COUNT_IF(average_playtime_2weeks IS NULL) AS average_playtime_2weeks,\n    COUNT_IF(median_playtime_2weeks IS NULL) AS median_playtime_2weeks,\n    COUNT_IF(short_description IS NULL) AS short_description,\n    COUNT_IF(about_the_game IS NULL) AS about_the_game,\n    COUNT_IF(detailed_description IS NULL) AS detailed_description,\n    COUNT_IF(notes IS NULL) AS notes,\n    COUNT_IF(achievements IS NULL) AS achievements,\n    COUNT_IF(dlc_count IS NULL) AS dlc_count,\n    COUNT_IF(packages IS NULL) AS packages,\n    COUNT_IF(supported_languages IS NULL) AS supported_languages,\n    COUNT_IF(full_audio_languages IS NULL) AS full_audio_languages,\n    COUNT_IF(windows IS NULL) AS windows,\n    COUNT_IF(mac IS NULL) AS mac,\n    COUNT_IF(linux IS NULL) AS linux,\n    COUNT_IF(steam_store_available IS NULL) AS steam_store_available,\n    COUNT_IF(steam_spy_available IS NULL) AS steam_spy_available,\n    COUNT_IF(_ingested_at IS NULL) AS _ingested_at,\n    COUNT_IF(_source IS NULL) AS _source\n  FROM steam_game_intelligence.bronze.steam_games\n) UNPIVOT (null_count FOR column_name IN (\n  app_id, name, release_date, price, price_status, estimated_owners,\n  developers, publishers, genres, categories, tags, positive, negative,\n  recommendations, peak_ccu, metacritic_score, user_score,\n  average_playtime_forever, median_playtime_forever, average_playtime_2weeks,\n  median_playtime_2weeks, short_description, about_the_game, detailed_description,\n  notes, achievements, dlc_count, packages, supported_languages,\n  full_audio_languages, windows, mac, linux, steam_store_available,\n  steam_spy_available, _ingested_at, _source\n))\nWHERE null_count > 0\nORDER BY null_count DESC;'), ('Consulta de celda SQL 3', 'SELECT app_id, COUNT(*) AS row_count,\n       COUNT(DISTINCT CAST(_ingested_at AS DATE)) AS ingestion_days,\n       MIN(_ingested_at) AS first_seen,\n       MAX(_ingested_at) AS last_seen\nFROM steam_game_intelligence.bronze.steam_games\nWHERE app_id IS NOT NULL\nGROUP BY app_id\nHAVING COUNT(*) > 1\nORDER BY row_count DESC\nLIMIT 100;'), ('Consulta de celda SQL 4', 'SELECT app_id, CAST(_ingested_at AS DATE) AS ingestion_date,\n       COUNT(*) AS row_count\nFROM steam_game_intelligence.bronze.steam_games\nGROUP BY app_id, CAST(_ingested_at AS DATE)\nHAVING COUNT(*) > 1\nORDER BY row_count DESC\nLIMIT 100;'), ('Consulta de celda SQL 5', 'SELECT\n  COUNT_IF(app_id <= 0) AS invalid_app_id,\n  COUNT_IF(price < 0) AS negative_price,\n  COUNT_IF(positive < 0) AS negative_positive_reviews,\n  COUNT_IF(negative < 0) AS negative_negative_reviews,\n  COUNT_IF(recommendations < 0) AS negative_recommendations,\n  COUNT_IF(peak_ccu < 0) AS negative_peak_ccu,\n  COUNT_IF(metacritic_score < 0 OR metacritic_score > 100) AS metacritic_out_of_range,\n  COUNT_IF(user_score < 0 OR user_score > 100) AS user_score_out_of_range,\n  COUNT_IF(achievements < 0) AS negative_achievements,\n  COUNT_IF(dlc_count < 0) AS negative_dlc_count,\n  COUNT_IF(release_date > CURRENT_DATE()) AS future_release_date,\n  COUNT_IF(_ingested_at IS NULL) AS missing_ingestion_timestamp\nFROM steam_game_intelligence.bronze.steam_games;'), ('Consulta de celda SQL 6', "SELECT app_id, name, release_date, price, positive, negative,\n       recommendations, peak_ccu, metacritic_score, user_score,\n       achievements, dlc_count, _ingested_at\nFROM steam_game_intelligence.bronze.steam_games\nWHERE app_id <= 0\n   OR price < 0\n   OR positive < 0 OR negative < 0\n   OR recommendations < 0 OR peak_ccu < 0\n   OR metacritic_score < 0 OR metacritic_score > 100\n   OR user_score < 0 OR user_score > 100\n   OR achievements < 0 OR dlc_count < 0\n   OR release_date > CURRENT_DATE()\n   OR name IS NULL OR TRIM(name) = ''\nLIMIT 200;"), ('Consulta de celda SQL 7', "SELECT\n  COUNT_IF(name IS NOT NULL AND TRIM(name) = '') AS blank_name,\n  COUNT_IF(developers IS NOT NULL AND TRIM(developers) = '') AS blank_developers,\n  COUNT_IF(publishers IS NOT NULL AND TRIM(publishers) = '') AS blank_publishers,\n  COUNT_IF(genres IS NOT NULL AND TRIM(genres) = '') AS blank_genres,\n  COUNT_IF(categories IS NOT NULL AND TRIM(categories) = '') AS blank_categories,\n  COUNT_IF(_source IS NOT NULL AND TRIM(_source) = '') AS blank_source\nFROM steam_game_intelligence.bronze.steam_games;"), ('Consulta de celda SQL 8', 'SELECT app_id,\n       COUNT(DISTINCT name) AS distinct_names,\n       COUNT(DISTINCT price) AS distinct_prices,\n       COUNT(DISTINCT release_date) AS distinct_release_dates,\n       COUNT(DISTINCT genres) AS distinct_genre_values,\n       COUNT(*) AS snapshots\nFROM steam_game_intelligence.bronze.steam_games\nWHERE app_id IS NOT NULL\nGROUP BY app_id\nHAVING COUNT(DISTINCT name) > 1\n    OR COUNT(DISTINCT price) > 1\n    OR COUNT(DISTINCT release_date) > 1\n    OR COUNT(DISTINCT genres) > 1\nORDER BY snapshots DESC\nLIMIT 100;')]
queries.append(("Muestra de estructura (una fila)", f"SELECT * FROM {table_name} LIMIT 1"))

def markdown_value(value):
    if value is None:
        return "NULL"
    return str(value).replace("|", "\\|").replace("\r", " ").replace("\n", " ")

report = [
    f"# Resultados del diagn?stico ? {notebook_name}",
    "",
    f"- **Notebook:** `{notebook_name}`",
    f"- **Ejecutado:** `{datetime.now().astimezone().isoformat()}` (zona horaria del driver)",
    "- **Estado:** `completed`",
    "- **Objetivo:** conservar los resultados efectivos de las validaciones SQL del notebook para su revisi?n local.",
    "",
    "## Tablas inspeccionadas",
    f"- `{table_name}`",
    "",
    "## Consultas y resultados",
]
for title, query in queries:
    rows = [row.asDict(recursive=True) for row in spark.sql(query).limit(200).collect()]
    report.extend(["", f"### {title}", f"_Filas exportadas: {len(rows)}; m?ximo aplicado: 200._", ""])
    if not rows:
        report.append("No se detectaron registros que incumplan esta validaci?n o la consulta no devolvi? filas.")
        continue
    columns = list(rows[0].keys())
    report.append("| " + " | ".join(columns) + " |")
    report.append("| " + " | ".join(["---"] * len(columns)) + " |")
    for row in rows:
        report.append("| " + " | ".join(markdown_value(row.get(column)) for column in columns) + " |")
report.extend(["", "## Hallazgos y conclusiones", "Los hallazgos deben interpretarse a partir de los conteos y muestras exportados; esta celda no inventa conclusiones ni marca como aprobada una validaci?n no ejecutada.", ""])

context_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
marker = "/src/transformation/diagnosticos"
if marker not in context_path:
    raise RuntimeError(f"No se pudo determinar la ra?z del repositorio desde la ruta del notebook: {context_path}")
repo_path = context_path.split(marker, 1)[0]
workspace_repo_path = repo_path if repo_path.startswith("/Workspace") else "/Workspace" + repo_path
output_dir = os.path.join(workspace_repo_path, "execution_results")
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, f"resultados_{notebook_name}.md")
temp_path = output_path + ".tmp"
with open(temp_path, "w", encoding="utf-8", newline="\n") as handle:
    handle.write("\n".join(report))
os.replace(temp_path, output_path)
print(f"Markdown generado: {output_path}; consultas exportadas: {len(queries)}")
